In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from decouple import config
OPENROUTER_API_KEY = config("OPENROUTER_API_KEY")

In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    model_name="openrouter/owl-alpha"
)

In [5]:
llm.invoke("hi")

AIMessage(content='Hello! How can I help you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 94, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'openrouter/owl-alpha', 'system_fingerprint': None, 'id': 'gen-1781586618-j0SWV1urzJcQJecllbDz', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eced5-9ec8-7602-83b7-d055f4943128-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 12, 'total_tokens': 106, 'input_token_details':

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/Chinook.db")

In [7]:
cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

cursor.fetchall()

[('Album',),
 ('Artist',),
 ('Customer',),
 ('Employee',),
 ('Genre',),
 ('Invoice',),
 ('InvoiceLine',),
 ('MediaType',),
 ('Playlist',),
 ('PlaylistTrack',),
 ('Track',)]

In [8]:
def get_schema(conn):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """)

    tables = cursor.fetchall()
    schema = ""

    for table in tables:
        table_name = table[0]
        cursor.execute(
            f"PRAGMA table_info({table_name})"
        )
        columns = cursor.fetchall()
        schema += f"\nTABLE {table_name}\n"

        for col in columns:
            column_name = col[1]
            datatype = col[2]
            schema += (
                f"  {column_name} {datatype}\n"
            )

    return schema

In [9]:
schema = get_schema(conn)

print(schema[:3000])


TABLE Album
  AlbumId INTEGER
  Title NVARCHAR(160)
  ArtistId INTEGER

TABLE Artist
  ArtistId INTEGER
  Name NVARCHAR(120)

TABLE Customer
  CustomerId INTEGER
  FirstName NVARCHAR(40)
  LastName NVARCHAR(20)
  Company NVARCHAR(80)
  Address NVARCHAR(70)
  City NVARCHAR(40)
  State NVARCHAR(40)
  Country NVARCHAR(40)
  PostalCode NVARCHAR(10)
  Phone NVARCHAR(24)
  Fax NVARCHAR(24)
  Email NVARCHAR(60)
  SupportRepId INTEGER

TABLE Employee
  EmployeeId INTEGER
  LastName NVARCHAR(20)
  FirstName NVARCHAR(20)
  Title NVARCHAR(30)
  ReportsTo INTEGER
  BirthDate DATETIME
  HireDate DATETIME
  Address NVARCHAR(70)
  City NVARCHAR(40)
  State NVARCHAR(40)
  Country NVARCHAR(40)
  PostalCode NVARCHAR(10)
  Phone NVARCHAR(24)
  Fax NVARCHAR(24)
  Email NVARCHAR(60)

TABLE Genre
  GenreId INTEGER
  Name NVARCHAR(120)

TABLE Invoice
  InvoiceId INTEGER
  CustomerId INTEGER
  InvoiceDate DATETIME
  BillingAddress NVARCHAR(70)
  BillingCity NVARCHAR(40)
  BillingState NVARCHAR(40)
  BillingC

In [ ]:
SQL_PROMPT = """
You are an expert SQLite analyst.
Your job is to convert a business question into a valid SQLite query.

DATABASE SCHEMA:
{schema}

INSTRUCTIONS:

1. Generate ONLY a valid SQLite SELECT query.
2. Do NOT return markdown, code fences, or explanations.
3. Do NOT include any text before or after the SQL query.
4. Use ONLY tables and columns present in the schema.
5. Never invent table names, column names, or relationships.
6. Use JOINs when information spans multiple tables.
7. Use appropriate aggregations (SUM, COUNT, AVG, MIN, MAX) when needed.
8. If the question is not in English, translate it internally and generate SQL.
9. If the question cannot be answered using the schema, return exactly: SCHEMA_ERROR
10. If multiple interpretations are possible, choose the most reasonable business interpretation.
11. Never generate DML queries.
12. Assume the user wants analytical insights unless explicitly stated otherwise.

OUTPUT FORMAT:
- Return only SQL
OR
- Return exactly: SCHEMA_ERROR

QUESTION:
{question}
"""

In [11]:
from langchain_core.messages import HumanMessage

In [22]:
def generate_sql(question, schema):
    prompt = SQL_PROMPT.format(
        schema=schema,
        question=question
    )
    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    return response.content.strip()

In [13]:
question = """
Show top 10 customers by revenue
"""

sql_query = generate_sql(
    question,
    schema
)

print(sql_query)

SELECT c.CustomerId, c.FirstName, c.LastName, SUM(i.Total) AS TotalRevenue
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName
ORDER BY TotalRevenue DESC
LIMIT 10


In [17]:
question = """
बिक्री के हिसाब से शीर्ष कलाकार
"""

sql_query = generate_sql(
    question,
    schema
)

print(sql_query)

SELECT ar.Name AS ArtistName, SUM(il.UnitPrice * il.Quantity) AS TotalSales
FROM Artist ar
JOIN Album al ON ar.ArtistId = al.ArtistId
JOIN Track t ON al.AlbumId = t.AlbumId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY ar.ArtistId, ar.Name
ORDER BY TotalSales DESC
LIMIT 5;


In [23]:
question = """
ਸਭ ਤੋਂ ਵੱਧ ਵਿਕਣ ਵਾਲੀਆਂ ਸੰਗੀਤ ਸ਼ੈਲੀਆਂ
"""

sql_query = generate_sql(
    question,
    schema
)

print(sql_query)

SELECT g.Name AS Genre, SUM(il.Quantity) AS TotalSales
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY g.GenreId, g.Name
ORDER BY TotalSales DESC
LIMIT 5;


In [ ]:
question = """
أي الموظفين يقدمون أكبر عدد من الدعم للعملاء؟
"""

sql_query = generate_sql(
    question,
    schema
)

print(sql_query)

SELECT e.EmployeeId, e.FirstName, e.LastName, COUNT(c.CustomerId) AS CustomerCount
FROM Employee e
JOIN Customer c ON e.EmployeeId = c.SupportRepId
GROUP BY e.EmployeeId, e.FirstName, e.LastName
ORDER BY CustomerCount DESC;


In [14]:
FORBIDDEN = [
    "DROP",
    "DELETE",
    "UPDATE",
    "INSERT",
    "ALTER",
    "TRUNCATE"
]

def validate_sql(sql):
    sql_upper = sql.upper()
    
    for keyword in FORBIDDEN:
        if keyword in sql_upper:
            raise ValueError(
                f"Forbidden SQL detected: {keyword}"
            )

    return True

In [15]:
def execute_sql(sql, conn):
    validate_sql(sql)
    df = pd.read_sql_query(
        sql,
        conn
    )
    return df

In [16]:
df = execute_sql(
    sql_query,
    conn
)

df.head()

,CustomerId,FirstName,LastName,TotalRevenue
0,6,Helena,Holý,49.62
1,26,Richard,Cunningham,47.62
2,57,Luis,Rojas,46.62
3,45,Ladislav,Kovács,45.62
4,46,Hugh,O'Reilly,45.62
